# 📝 과제 02: 온라인 쇼핑 상담 어댑터와 양자화 모델 서빙

## 만들 결과

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">교안 02의 어댑터·양자화 서빙</strong>을 쇼핑 상담에 적용합니다. 모델을 다시 학습하지 않고 저장한 어댑터와 체크포인트를 사용합니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">쇼핑·노무 어댑터의 모델 목록 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model_table</code>과 답변 비교 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">adapter_table</code></li>
<li style="margin:3px 0;">원본·GPTQ·AWQ의 양자화 방식과 메모리 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quant_table</code></li>
<li style="margin:3px 0;">같은 쇼핑 질문 3개에 대한 세 모델의 답변 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quant_answer_table</code></li>
</ul>
</div>

## 준비: 데이터와 제공 코드

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">선행 실습:</strong> <strong style="font-size:1.13em;color:#1f3a5f;">RunPod에</strong> 교안 02의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/qwen3.5-0.8b-text</code>가 있어야 합니다. 이 노트북은 내 PC에서 실행하고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 공개 주소·API 키를 사용합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">자료:</strong> LoRA 단원의 쇼핑·노무 어댑터와 SFT 단원의 쇼핑 평가 질문을 사용합니다. 어댑터 비교는 3문항, 양자화 비교도 같은 평가셋의 앞 3문항입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">제공 코드:</strong> PC에서 경로·평가 질문·어댑터 설정 파일을 준비합니다. 어댑터 가중치 준비와 서버 실행은 RunPod 터미널 명령으로 진행합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성:</strong> 요구사항 번호에 맞춰 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>를 채웁니다. 모델을 바꾸기 전에 현재 서버를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 종료합니다.</li>
</ul>
</div>

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from openai import OpenAI
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
material_dir = Path(".").resolve()
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)
day70_dir = material_dir.parent / "day70_SFT_지도파인튜닝"
SERVER_URL = os.environ["VLLM_SERVER_URL"].rstrip("/")  # RunPod 공개 HTTPS 주소
VLLM_API_KEY = os.environ["VLLM_API_KEY"]
MAX_TOKENS = 256     # 답변으로 생성할 최대 토큰 수
pd.set_option("display.max_colwidth", None)

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(기존 실습과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
SYSTEM_PROMPT_COMMERCE = "너는 온라인 쇼핑 상담 챗봇이다. 전자상거래법 조문에 근거해 존댓말로 간결히 답한다."
eval_rows = load_jsonl(day70_dir / "data" / "과제" / "commerce_eval.jsonl")

In [ ]:
from huggingface_hub import hf_hub_download

BASE_MODEL = "Qwen/Qwen3.5-0.8B"
commerce_config_path = hf_hub_download("muahseol/enkoa-qwen3.5-0.8b-commerce-lora", "adapter_config.json")
labor_config_path = hf_hub_download("muahseol/enkoa-qwen3.5-0.8b-labor-lora", "adapter_config.json")

## 1. 어댑터 설정 확인

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 같은 베이스 모델에 연결할 수 있는 어댑터인지 확인하고 서버가 지원해야 할 최소 rank를 구합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce_config_path</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_config_path</code>의 JSON을 읽어 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">adapter_config</code></strong>(쇼핑), <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">labor_adapter_config</code></strong>(노무)에 담습니다.</li>
<li style="margin:3px 0;">(2) 두 설정의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">r</code> 중 큰 값을 정수 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">required_lora_rank</code></strong>에 담습니다. 제공 코드가 베이스 모델·rank·DoRA 여부를 표시합니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 베이스 모델과 어댑터 준비를 참고합니다.
JSON 파일을 json.load로 읽습니다. 각 설정의 r을 꺼내 max로 비교합니다.
```

</details>

In [ ]:
# TODO (1): 두 어댑터의 설정 파일을 읽습니다.
with open(commerce_config_path, encoding="utf-8") as f:
    adapter_config = ...
with open(labor_config_path, encoding="utf-8") as f:
    labor_adapter_config = ...
# TODO (2): 사용할 두 어댑터 중 가장 큰 rank를 구합니다.
required_lora_rank = ...
adapter_settings = pd.DataFrame([
    {"어댑터":"commerce-lora", "베이스 모델":adapter_config["base_model_name_or_path"],
     "r":adapter_config["r"], "use_dora":adapter_config["use_dora"]},
    {"어댑터":"labor-lora", "베이스 모델":labor_adapter_config["base_model_name_or_path"],
     "r":labor_adapter_config["r"], "use_dora":labor_adapter_config["use_dora"]},
])
display(adapter_settings)
print("필요한 최소 rank:", required_lora_rank)

In [ ]:
# [자가채점] 베이스 모델과 사용할 rank를 확인합니다.
assert adapter_config["base_model_name_or_path"] == labor_adapter_config["base_model_name_or_path"] == BASE_MODEL, "지정된 두 어댑터의 설정을 읽으세요."
assert not adapter_config["use_dora"] and not labor_adapter_config["use_dora"], "이 서버에는 LoRA 어댑터를 연결합니다."
assert isinstance(required_lora_rank,int) and required_lora_rank == 8, "이 과제의 두 어댑터는 r=8입니다. 읽은 설정을 확인하세요."
print("1번 완료")

## 2. 어댑터 서버 실행과 모델 목록

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 베이스 한 벌에 쇼핑·노무 어댑터를 연결하고 API에서 선택할 이름을 확인합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널</strong>에서 두 어댑터의 가중치를 준비한 뒤 서버를 실행합니다. 베이스 폴더는 교안 02에서 만든 것을 사용합니다. <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--max-loras 2</code></strong>는 쇼핑·노무 어댑터 두 종류를 함께 처리하도록, <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--max-lora-rank 8</code></strong>은 1번에서 확인한 어댑터의 rank에 맞춰 지정합니다.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습자료 폴더에서 새 RunPod 터미널</strong>을 열고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day75_vLLM_추론엔진_서빙</code> 폴더로 이동합니다. 서버 명령은 이 폴더의 RunPod 전용 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv.lock</code> 환경을 사용합니다. 부록 01에서 자동 생성하여 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 저장한 API 키를 입력하고, 이 터미널에서 서버를 계속 실행합니다.</p>
</div>

```bash
cd day75_vLLM_추론엔진_서빙
mkdir -p output
read -s -p "서버 API 키 입력: " VLLM_API_KEY
export VLLM_API_KEY
```

**RunPod 터미널 · 파일 준비**

```bash
uv run python - <<'PY'
from pathlib import Path
from huggingface_hub import snapshot_download

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
snapshot_download('muahseol/enkoa-qwen3.5-0.8b-commerce-lora', local_dir=output_dir / 'commerce_lora_adapter')
snapshot_download('muahseol/enkoa-qwen3.5-0.8b-labor-lora', local_dir=output_dir / 'labor_lora_adapter')
print("서버 파일 준비 완료")
PY
```

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve output/qwen3.5-0.8b-text \
  --served-model-name qwen3.5-0.8b \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --enable-lora \
  --lora-modules commerce-lora=output/commerce_lora_adapter labor-lora=output/labor_lora_adapter \
  --max-loras 2 \
  --max-lora-rank 8 \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/commerce_lora_server.log
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 요청을 보낼 수 있습니다. RunPod 터미널은 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 노트북</strong>으로 돌아옵니다. 첫 실행은 모델 다운로드와 GPU 계산 준비에 시간이 걸릴 수 있습니다.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVER_URL</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code> 주소와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>를 사용하는 OpenAI <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">client</code></strong>를 만듭니다.</li>
<li style="margin:3px 0;">(2) 제공 코드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model_rows</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">id</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">root</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">parent</code> 열 순서로 가진 테이블 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">model_table</code></strong>로 만듭니다.</li>
</ul>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 모델 목록과 어댑터를 참고합니다.
부를 수 있는 모델 이름은 id입니다. parent는 어댑터가 연결된 베이스 서버 이름을 나타냅니다.
```

</details>

In [ ]:
# TODO (1): 어댑터 서버에 OpenAI 클라이언트를 연결합니다.
client = ...
model_rows = [model.to_dict() for model in client.models.list()]
# TODO (2): 세 열을 가진 테이블을 만듭니다.
model_table = ...
display(model_table)

In [ ]:
# [자가채점] 등록된 모델과 어댑터의 연결을 확인합니다. 행 순서는 자유입니다.
assert list(model_table.columns) == ["id","root","parent"], "요구사항의 세 열을 선택하세요."
assert set(model_table["id"]) == {"qwen3.5-0.8b","commerce-lora","labor-lora"}, "터미널 명령의 모델과 어댑터 이름을 확인하세요."
parents = model_table.set_index("id")["parent"]
assert parents["commerce-lora"] == parents["labor-lora"] == "qwen3.5-0.8b", "parent 값을 그대로 담으세요."
assert pd.isna(parents["qwen3.5-0.8b"]), "베이스 모델의 parent는 비어 있습니다."
print("2번 완료")

## 3. model 이름으로 쇼핑 어댑터 선택

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 같은 서버에서 요청의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code>만 바꿔 베이스 모델과 쇼핑 어댑터를 선택합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">eval_rows[:3]</code>마다 제공된 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">messages</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-0.8b</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce-lora</code>에 보내 각각 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">base_response</code></strong>, <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">adapter_response</code></strong>에 담습니다. 두 요청 모두 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">temperature=0</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_completion_tokens=MAX_TOKENS</code>입니다.</li>
<li style="margin:3px 0;">(2) 각 응답의 첫 선택지 답 문자열을 행의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Base 답변</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">commerce-lora 답변</code> 칸에 담습니다. 제공 코드가 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">adapter_table</code></strong>을 표시합니다.</li>
</ul>
<p style="margin:6px 0;">같은 답이 나올 수도 있습니다. 답변의 문자열 차이를 어댑터 적용의 채점 조건으로 삼지 않습니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 model 이름으로 어댑터 선택을 참고합니다.
두 요청은 model만 다릅니다. choices 첫 항목의 message.content를 각 답변 칸에 담습니다.
```

</details>

In [ ]:
adapter_rows = []
for row in eval_rows[:3]:
    messages = [{"role":"system","content":SYSTEM_PROMPT_COMMERCE}, {"role":"user","content":row["question"]}]
    # TODO (1): 같은 질문을 베이스 모델과 쇼핑 어댑터에 보냅니다.
    base_response = ...
    adapter_response = ...
    # TODO (2): 두 응답의 답 문자열을 각각 담습니다.
    adapter_rows.append({
        "질문":row["question"], "정답 답변":row["reference_answer"],
        "Base 답변":..., "commerce-lora 답변":...,
    })
adapter_table = pd.DataFrame(adapter_rows)
display(adapter_table)

In [ ]:
# [자가채점] 요청 모델과 세 질문의 응답을 확인합니다.
assert base_response.model == "qwen3.5-0.8b" and adapter_response.model == "commerce-lora", "요청에서 선택한 model 이름을 확인하세요."
assert list(adapter_table.columns) == ["질문","정답 답변","Base 답변","commerce-lora 답변"]
assert list(adapter_table["질문"]) == [row["question"] for row in eval_rows[:3]], "앞 3개 질문의 순서를 유지하세요."
for column in ["Base 답변","commerce-lora 답변"]:
    assert all(isinstance(value,str) and value.strip() for value in adapter_table[column]), "응답의 답 문자열을 담으세요."
print("3번 완료")

### 어댑터 서버 종료와 GPU 메모리 반환

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod에서 서버를 실행한 터미널</strong>로 돌아가 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code></strong>를 누릅니다. 프롬프트가 돌아오면 같은 터미널에서 확인합니다.</p>
</div>

```bash
nvidia-smi
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 서버 프로세스가 사라지고 GPU 메모리 사용량이 실행 전 수준으로 줄었는지</strong> 확인합니다. 서버 프로세스가 종료되면 모델 가중치와 KV 캐시 메모리도 반환됩니다. 확인을 마친 뒤 다음 단계로 이동합니다.</p>
</div>

## 4. 원본·GPTQ·AWQ 모델 서빙

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 같은 질문을 세 체크포인트에 보내고 서버가 인식한 양자화 방식과 메모리 사용량을 기록합니다.</p>
<p style="margin:6px 0;">어댑터 서버 종료와 메모리 반환 확인을 마쳤다면 진행합니다. 다음 두 준비 셀은 <strong style="font-size:1.13em;color:#1f3a5f;">처음 한 번만</strong> 실행합니다.</p>
</div>

In [ ]:
quant_models = [{'모델': '원본 bf16', '저장소': 'Qwen/Qwen3.5-4B', '서버 이름': 'qwen3.5-4b'},
 {'모델': 'GPTQ W4A16',
  '저장소': 'RedHatAI/Qwen3.5-4B-quantized.w4a16',
  '서버 이름': 'qwen3.5-4b-gptq'},
 {'모델': 'AWQ 4비트', '저장소': 'QuantTrio/Qwen3.5-4B-AWQ', '서버 이름': 'qwen3.5-4b-awq'}]
quant_results = {}  # 모델 이름별로 저장합니다. 같은 모델을 다시 실행하면 그 결과를 덮어씁니다.

In [ ]:
quant_questions = [row["question"] for row in eval_rows[:3]]
quant_message_lists = []
for question in quant_questions:
    quant_message_lists.append([("system", SYSTEM_PROMPT_COMMERCE), ("human", question)])

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">아래 명령은 <strong style="font-size:1.13em;color:#1f3a5f;">한 번에 하나만</strong> 터미널에서 실행합니다. 준비 완료 로그를 확인하고 작성 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model_index</code>를 해당 번호로 바꿔 결과를 모읍니다. 다음 모델을 실행하기 전에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 현재 서버를 종료합니다.</p>
<p style="margin:6px 0;">각 모델이 준비되면 RunPod의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/commerce_&lt;서버 이름&gt;_server.log</code>를 내려받아 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">output</code></strong>에 같은 이름으로 저장합니다. API 요청은 PC에서 공개 주소로 보냅니다.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--language-model-only</code>는 이미지·음성 입력을 끄고 텍스트 요청만 받는 설정입니다. 나머지 서버 옵션은 앞 실습과 같습니다.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">0: 원본 bf16</strong></p>
</div>

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve Qwen/Qwen3.5-4B \
  --served-model-name qwen3.5-4b \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/commerce_qwen3.5-4b_server.log
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">1: GPTQ W4A16</strong></p>
</div>

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve RedHatAI/Qwen3.5-4B-quantized.w4a16 \
  --served-model-name qwen3.5-4b-gptq \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/commerce_qwen3.5-4b-gptq_server.log
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">2: AWQ 4비트</strong></p>
</div>

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve QuantTrio/Qwen3.5-4B-AWQ \
  --served-model-name qwen3.5-4b-awq \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/commerce_qwen3.5-4b-awq_server.log
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) 현재 모델의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">log_text</code>에서 가중치 메모리를 실수 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">weight_gib</code></strong>, KV 캐시 토큰 수를 정수 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">kv_cache_tokens</code></strong>에 담습니다.</li>
<li style="margin:3px 0;">(2) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quantization=&lt;형식&gt;</code>에서 형식 문자열을 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">quant_method</code></strong>에 담습니다. 원본 모델도 로그에 기록된 값을 그대로 읽습니다.</li>
<li style="margin:3px 0;">(3) 현재 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quant_model[&quot;서버 이름&quot;]</code>을 호출하는 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">quant_llm</code></strong>을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatOpenAI</code>로 만듭니다. 주소·키는 2번과 같고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">temperature=0</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_tokens=MAX_TOKENS</code>입니다.</li>
<li style="margin:3px 0;">(4) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quant_message_lists</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">batch</code>로 보내 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">quant_answers</code></strong>에 담습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_concurrency</code>는 3입니다. 제공 코드가 결과를 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">quant_results</code></strong>에 모델 이름별로 저장합니다.</li>
</ul>
<p style="margin:6px 0;">아래 두 작성 셀과 자가채점은 <strong style="font-size:1.13em;color:#1f3a5f;">모델마다 실행</strong>합니다. 결과를 모으는 동안 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quant_results</code>를 초기화하지 않습니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 현재 모델의 로그와 답변 수집을 참고합니다.
0은 원본, 1은 GPTQ, 2는 AWQ입니다. 로그 값은 re.search의 group(1)로 꺼냅니다. quantization 형식은 숫자가 아닌 문자열입니다.
```

</details>

In [ ]:
model_index = 0  # 터미널에서 실행한 모델에 맞게 0 / 1 / 2로 바꿉니다.
quant_model = quant_models[model_index]
log_name = f"commerce_{quant_model['서버 이름']}_server.log"
log_text = (output_dir / log_name).read_text(encoding="utf-8")
# TODO (1): 가중치 메모리와 KV 캐시 용량을 로그에서 읽습니다.
weight_gib = ...
kv_cache_tokens = ...
# TODO (2): 서버가 인식한 양자화 형식을 읽습니다.
quant_method = ...

In [ ]:
# TODO (3): 현재 모델을 부르는 LangChain 클라이언트를 만듭니다.
quant_llm = ...
# TODO (4): 질문 세 개를 한꺼번에 보냅니다.
quant_answers = ...
quant_results[quant_model["모델"]] = {
    "모델":quant_model["모델"], "양자화 방식":quant_method,
    "가중치 메모리(GiB)":weight_gib, "KV 캐시(토큰)":kv_cache_tokens,
    "답변":[answer.content for answer in quant_answers],
}
display(quant_results[quant_model["모델"]])

In [ ]:
# [자가채점] 현재 모델 한 개의 로그와 응답을 확인합니다.
assert isinstance(weight_gib,float) and weight_gib > 0, "가중치 GiB를 실수로 읽으세요."
assert isinstance(kv_cache_tokens,int) and kv_cache_tokens > 0, "KV 캐시 토큰 수를 정수로 읽으세요."
expected_weight_gib = float(re.search(r"Model loading took ([\d.]+) GiB", log_text).group(1))
expected_cache_tokens = int(re.search(r"GPU KV cache size: ([\d,]+) tokens", log_text).group(1).replace(",", ""))
expected_quant_method = re.search(r"quantization=([\w-]+)", log_text).group(1)
assert weight_gib == expected_weight_gib, "가중치 메모리를 현재 모델의 로그에서 읽으세요."
assert kv_cache_tokens == expected_cache_tokens, "KV 캐시 토큰 수를 현재 모델의 로그에서 읽으세요."
assert quant_method == expected_quant_method, "quantization 뒤의 형식 이름 전체를 읽으세요."
assert quant_llm.model_name == quant_model["서버 이름"], "현재 서버의 모델 이름으로 요청하세요."
assert len(quant_answers) == 3, "같은 질문 3개에 대한 응답을 받아야 합니다."
assert all(isinstance(value.content,str) and value.content.strip() for value in quant_answers), "빈 응답이 있는지 확인하세요."
print(quant_model["모델"], "수집 완료")

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">현재 모델의 수집이 끝났다면 · 원본/GPTQ/AWQ 공통</strong></p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod에서 서버를 실행한 터미널</strong>로 돌아가 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code></strong>를 누릅니다. 프롬프트가 돌아오면 같은 터미널에서 확인합니다.</p>
</div>

```bash
nvidia-smi
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 서버 프로세스가 사라지고 GPU 메모리 사용량이 실행 전 수준으로 줄었는지</strong> 확인합니다. 서버 프로세스가 종료되면 모델 가중치와 KV 캐시 메모리도 반환됩니다. 확인을 마친 뒤 다음 단계로 이동합니다.</p>
<p style="margin:6px 0;">다음 모델을 실행하고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model_index</code>를 바꿔 결과 수집·자가채점 셀을 다시 실행합니다. 세 모델을 모두 마쳤다면 5번으로 이동합니다.</p>
</div>

## 5. 세 모델의 결과 정리

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 같은 질문에 대한 메모리와 답변을 한 화면에서 비교할 수 있게 정리합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) 제공 코드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ordered_results</code>로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">quant_table</code></strong>을 만듭니다. 열 순서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">모델</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">양자화 방식</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">가중치 메모리(GiB)</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">KV 캐시(토큰)</code>입니다.</li>
<li style="margin:3px 0;">(2) 제공 코드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer_columns</code>로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">quant_answer_table</code></strong>을 만듭니다. 열은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">질문</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답 답변</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">원본 답변</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPTQ 답변</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">AWQ 답변</code> 순서입니다.</li>
</ul>
<p style="margin:6px 0;">답변 문구가 달라지거나 KV 캐시 용량이 예상과 다를 수 있습니다. 문자열 일치 여부나 메모리 감소 비율은 채점하지 않습니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 메모리와 답변 비교를 참고합니다.
dict의 수집 순서와 관계없이 ordered_results가 원본·GPTQ·AWQ 순서를 맞춥니다. DataFrame을 만들고 요구사항의 열만 선택합니다.
```

</details>

In [ ]:
ordered_results = [quant_results[model["모델"]] for model in quant_models]
# TODO (1): 메모리 테이블을 만들고 네 열을 선택합니다.
quant_table = ...
display(quant_table.round(2))

answer_columns = {
    "질문":quant_questions,
    "정답 답변":[row["reference_answer"] for row in eval_rows[:3]],
    "원본 답변":quant_results["원본 bf16"]["답변"],
    "GPTQ 답변":quant_results["GPTQ W4A16"]["답변"],
    "AWQ 답변":quant_results["AWQ 4비트"]["답변"],
}
# TODO (2): 질문별 답변 비교 테이블을 만듭니다.
quant_answer_table = ...
display(quant_answer_table)

In [ ]:
# [자가채점] 세 모델의 수집 여부와 결과 구조를 확인합니다.
assert set(quant_results) == {"원본 bf16","GPTQ W4A16","AWQ 4비트"}, "세 모델을 하나씩 실행해 결과를 모으세요."
assert list(quant_table.columns) == ["모델","양자화 방식","가중치 메모리(GiB)","KV 캐시(토큰)"]
assert list(quant_table["모델"]) == [model["모델"] for model in quant_models], "원본·GPTQ·AWQ 순서로 정리하세요."
assert quant_answer_table.shape == (3,5), "질문 3개와 다섯 열을 가진 테이블입니다."
assert list(quant_answer_table.columns) == ["질문","정답 답변","원본 답변","GPTQ 답변","AWQ 답변"]
assert list(quant_answer_table["질문"]) == quant_questions, "원래 질문 순서를 유지하세요."
print("5번 완료")

### 마무리 확인

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">마지막 AWQ 서버의 <strong style="font-size:1.13em;color:#1f3a5f;">종료와 GPU 메모리 반환 확인</strong>까지 마쳤다면 실습 완료입니다.</p>
</div>